# GenAIScope Complete Google Colab Test Notebook

This notebook helps you test **GenAIScope v0.7.0** end-to-end from Google Colab (or any
local Jupyter kernel).

It covers:

- Installation from GitHub `main`, a Git tag, PyPI, or a local working copy
- Import/version validation
- Package structure inspection
- Core `Inspector` API smoke tests
- Analyzer tests: PII, cost, hallucination, safety, JSON validation
- Local memory tests
- Prompt coach tests
- Semantic cache tests
- File memory tests
- Local tracing tests
- CLI command tests
- Static dashboard generation
- **Context Doctor (v0.6.0)**: diagnose, context builder, cost/router, analytics, HTML report
- **MCP tools (v0.7.0)**: `doctor_diagnose`, `analytics_*`, `report_generate`, memory tools
- **Live LLM gateway (v0.7.0)**: `scope.gateway` / `genaiscope ask` auto-routing
- **Cross-encoder reranking (v0.7.0)**: `rerank=True` on memory search
- **Agent evaluation (v0.7.0)**: `genaiscope.evals.run_agent_eval`
- **LangChain / LlamaIndex integrations (v0.7.0)**
- **Langfuse batch export (v0.7.0)**
- **OpenTelemetry exporter hook (v0.7.0)**
- **Browser extension structure check (v0.7.0)**
- Repository test suite using `pytest`

No paid LLM API key is required for the default tests. The live gateway test only makes a
real network call if it finds a provider API key in the environment; otherwise it verifies
the expected `GatewayError` and skips.


## How to use this notebook

1. Open this `.ipynb` file in **Google Colab**, or run it locally with Jupyter/`nbconvert`.
2. Select **Runtime → Run all** (Colab) or execute all cells top to bottom (local).
3. Keep `INSTALL_SOURCE = "github"` to test the latest GitHub version.
4. Change `BRANCH_OR_TAG` to a release tag such as `v0.7.0` when you want to test a specific release.
5. Use `INSTALL_SOURCE = "pypi"` when you want to test the published PyPI package.
6. Use `INSTALL_SOURCE = "local"` to test the working copy this notebook lives in (no git
   clone, no network needed) — set `LOCAL_REPO_DIR` if the notebook isn't already inside the repo.

Expected success indicators:

- `import genaiscope` works
- version prints correctly
- core API examples run without error
- CLI commands respond
- memory/file/tracing/dashboard tests complete
- Context Doctor, MCP tools, gateway, reranking, agent-eval, and integration smoke tests complete
- `pytest` passes, if tests are present in the repository


In [1]:
# ============================================================
# 1. Global configuration
# ============================================================

import os
import sys
import json
import time
import shutil
import pathlib
import tempfile
import subprocess
import importlib
import pkgutil
from typing import Any, Dict, List, Optional

# Change these values when needed
REPO_URL = "https://github.com/TravelXML/GenAIScope.git"
BRANCH_OR_TAG = "main"       # examples: "main", "v0.7.0"
INSTALL_SOURCE = "local"     # options: "github", "pypi", "local"
PACKAGE_NAME = "genaiscope"

# Used only when INSTALL_SOURCE == "local": path to the working copy to install/test.
# Defaults to the current working directory, which is correct when this notebook is opened
# from inside the repo (the common case for local/CI runs).
LOCAL_REPO_DIR = pathlib.Path.cwd()

# /content only exists on Google Colab; fall back to a temp dir for local/CI execution.
_BASE_DIR = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path(tempfile.gettempdir())
WORKDIR = _BASE_DIR / "genaiscope_colab_test"
REPO_DIR = _BASE_DIR / "GenAIScope"
WORKDIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Workdir:", WORKDIR)
print("Repo URL:", REPO_URL)
print("Branch or tag:", BRANCH_OR_TAG)
print("Install source:", INSTALL_SOURCE)
if INSTALL_SOURCE == "local":
    print("Local repo dir:", LOCAL_REPO_DIR)


Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Workdir: /tmp/genaiscope_colab_test
Repo URL: https://github.com/TravelXML/GenAIScope.git
Branch or tag: main
Install source: local
Local repo dir: /home/sap-ai-plug/CODE/GenAIScope


In [2]:
# ============================================================
# 2. Helper functions
# ============================================================

def run_command(
    cmd: List[str],
    cwd: Optional[pathlib.Path] = None,
    check: bool = False,
    timeout: int = 180,
) -> subprocess.CompletedProcess:
    """Run a shell command safely and print stdout/stderr."""
    print("\n$", " ".join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(cmd)}")
    return result


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def safe_call(name: str, fn, *args, **kwargs) -> Dict[str, Any]:
    """Run a test function without stopping the notebook on failure."""
    section(name)
    try:
        value = fn(*args, **kwargs)
        print("✅ PASS:", name)
        return {"name": name, "status": "PASS", "value": repr(value)[:1000], "error": ""}
    except Exception as exc:
        print("❌ FAIL:", name)
        print(type(exc).__name__ + ":", exc)
        return {"name": name, "status": "FAIL", "value": "", "error": repr(exc)[:1000]}


test_results: List[Dict[str, Any]] = []


In [3]:
# ============================================================
# 3. Install GenAIScope
# ============================================================

section("Upgrade pip tooling")
run_command([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=False, timeout=240)

section("Install common test tools")
run_command([sys.executable, "-m", "pip", "install", "-U", "pytest", "pytest-cov", "pandas"], check=False, timeout=240)

if INSTALL_SOURCE == "github":
    section("Clone repository")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    clone_cmd = ["git", "clone", "--depth", "1", "--branch", BRANCH_OR_TAG, REPO_URL, str(REPO_DIR)]
    clone_result = run_command(clone_cmd, check=False, timeout=240)

    if clone_result.returncode != 0:
        print("Shallow clone failed. Retrying full clone and checkout...")
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run_command(["git", "clone", REPO_URL, str(REPO_DIR)], check=True, timeout=300)
        run_command(["git", "checkout", BRANCH_OR_TAG], cwd=REPO_DIR, check=True, timeout=120)

    section("Install from local cloned repository in editable mode (all extras)")
    editable_result = run_command([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[all,dev]"], check=False, timeout=300)

    if editable_result.returncode != 0:
        print("Editable install with extras failed. Retrying direct GitHub pip install...")
        run_command([sys.executable, "-m", "pip", "install", "-U", f"git+{REPO_URL}@{BRANCH_OR_TAG}"], check=True, timeout=300)

elif INSTALL_SOURCE == "pypi":
    section("Install from PyPI")
    run_command([sys.executable, "-m", "pip", "install", "-U", f"{PACKAGE_NAME}[all]"], check=True, timeout=300)

elif INSTALL_SOURCE == "local":
    section("Install from local working copy (editable, all extras)")
    REPO_DIR = LOCAL_REPO_DIR
    local_result = run_command([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[all,dev]"], check=False, timeout=300)
    if local_result.returncode != 0:
        print("Editable install with extras failed. Retrying without extras...")
        run_command([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True, timeout=300)

else:
    raise ValueError("INSTALL_SOURCE must be 'github', 'pypi', or 'local'")

# Move off the repo/clone directory for the rest of the notebook -- matches real Colab
# behavior (which never cds into /content/GenAIScope) and keeps cells that default to a
# cwd-relative db path (MemoryStore(), LocalTracer(), FileMemory()) from writing into the
# repo working copy when INSTALL_SOURCE == "local".
os.chdir(WORKDIR)
print("Working directory set to:", pathlib.Path.cwd())



Upgrade pip tooling

$ /home/sap-ai-plug/CODE/GenAIScope/.venv/bin/python3 -m pip install -U pip setuptools wheel


  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.12.1 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


Install common test tools

$ /home/sap-ai-plug/CODE/GenAIScope/.venv/bin/python3 -m pip install -U pytest pytest-cov pandas


  Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached pytest_cov-7.1.0-py3-none-any.whl.metadata (32 kB)
Using cached pytest-9.1.1-py3-none-any.whl (386 kB)
Using cached pytest_cov-7.1.0-py3-none-any.whl (22 kB)

  Attempting uninstall: pytest

    Found existing installation: pytest 8.4.2

    Uninstalling pytest-8.4.2:

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
      Successfully uninstalled pytest-8.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
  Attempting uninstall: pytest-cov
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
    Found existing installation: pytest-cov 5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
    Uninstalling pytest-cov-5.0.0:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [pytest]
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━

Obtaining file:///home/sap-ai-plug/CODE/GenAIScope
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached pytest-8.4.2-py3-none-any.whl.metadata (7.7 kB)
  Using cached pytest_cov-5.0.0-py3-none-any.whl.metadata (27 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached pytest-8.4.2-py3-none-any.whl (365 kB)
Using cached pytest_cov-5.0.0-py3-none-any.whl (21 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Building editable for genaiscope (pyproject.toml): started
  Building editable for genaisc

In [4]:
# ============================================================
# 4. Import and version check
# ============================================================

def import_and_version_check():
    pkg = importlib.import_module(PACKAGE_NAME)
    print("Package file:", getattr(pkg, "__file__", "unknown"))
    print("Package version:", getattr(pkg, "__version__", "unknown"))
    print("Top-level objects:", [x for x in dir(pkg) if not x.startswith("_")][:80])
    return getattr(pkg, "__version__", "unknown")

test_results.append(safe_call("Import package and print version", import_and_version_check))



Import package and print version
Package file: /home/sap-ai-plug/CODE/GenAIScope/src/genaiscope/__init__.py
Package version: 0.7.0
Top-level objects: ['ContextBuilder', 'ContextDoctor', 'CostEstimator', 'EvaluationResult', 'FileMemory', 'GenAIScope', 'InspectionReport', 'Inspector', 'LocalTracer', 'MemoryStore', 'Provider', 'Result', 'ResultStatus', 'ScopeConfig', 'ScoringEngine', 'analyzers', 'context', 'core', 'cost', 'dashboard', 'doctor', 'files', 'generate_dashboard', 'inspect', 'memory', 'scoring', 'tracing', 'vector']
✅ PASS: Import package and print version


In [5]:
# ============================================================
# 5. Inspect package modules
# ============================================================

def inspect_package_modules():
    pkg = importlib.import_module(PACKAGE_NAME)
    if not hasattr(pkg, "__path__"):
        print("Package has no __path__; cannot list modules.")
        return []

    modules = []
    for mod in pkgutil.iter_modules(pkg.__path__):
        modules.append({
            "name": mod.name,
            "is_package": mod.ispkg,
        })

    print(json.dumps(modules, indent=2))
    return modules

test_results.append(safe_call("Inspect package modules", inspect_package_modules))



Inspect package modules
[
  {
    "name": "adapters",
    "is_package": true
  },
  {
    "name": "analytics",
    "is_package": true
  },
  {
    "name": "analyzers",
    "is_package": false
  },
  {
    "name": "cache",
    "is_package": true
  },
  {
    "name": "cli",
    "is_package": true
  },
  {
    "name": "context",
    "is_package": true
  },
  {
    "name": "core",
    "is_package": true
  },
  {
    "name": "cost",
    "is_package": true
  },
  {
    "name": "dashboard",
    "is_package": true
  },
  {
    "name": "doctor",
    "is_package": true
  },
  {
    "name": "embeddings",
    "is_package": true
  },
  {
    "name": "evals",
    "is_package": true
  },
  {
    "name": "export",
    "is_package": true
  },
  {
    "name": "files",
    "is_package": true
  },
  {
    "name": "gateway",
    "is_package": true
  },
  {
    "name": "inspect",
    "is_package": false
  },
  {
    "name": "integrations",
    "is_package": true
  },
  {
    "name": "mcp",
    "is_package"

## Core Python API tests

The next cells test the main GenAIScope APIs. These are intentionally written as smoke tests, so they continue running even if one feature has changed in the latest code.


In [6]:
# ============================================================
# 6. Inspector smoke tests
# ============================================================

def test_inspector_api():
    from genaiscope import Inspector

    inspector = Inspector()

    print("\nPrompt inspection:")
    prompt_report = inspector.inspect_prompt("What is the capital of France?")
    print(prompt_report.summary() if hasattr(prompt_report, "summary") else prompt_report)

    print("\nRAG inspection:")
    rag_report = inspector.inspect_rag(
        query="What is AI?",
        context="Artificial Intelligence is a field of computer science focused on intelligent systems.",
        response="AI is a field that builds intelligent systems."
    )
    print(rag_report.summary() if hasattr(rag_report, "summary") else rag_report)

    print("\nStructured output inspection:")
    output_report = inspector.inspect_output('{"name": "GenAIScope", "type": "toolkit"}', expected_format="json")
    print(output_report.summary() if hasattr(output_report, "summary") else output_report)

    return "Inspector API completed"

test_results.append(safe_call("Inspector API smoke tests", test_inspector_api))


2026-07-01 12:13:26,910 - genaiscope.inspect - INFO - Inspecting prompt: What is the capital of France?...


2026-07-01 12:13:26,910 - genaiscope.inspect - INFO - Inspecting RAG system...


2026-07-01 12:13:26,911 - genaiscope.inspect - INFO - Inspecting output...



Inspector API smoke tests

Prompt inspection:
# Prompt Inspection

Analysis of prompt quality and potential issues

Timestamp: 2026-07-01 06:43:26.910836

## Evaluations
  - pass: 0.80
    Reasoning: Prompt structure looks reasonable

## Metrics
  - length: 30
  - word_count: 6
  - avg_word_length: 4.166666666666667

RAG inspection:
# RAG Inspection

Analysis of RAG system quality and context relevance

Timestamp: 2026-07-01 06:43:26.911495

## Evaluations
  - pass: 0.42
    Reasoning: Context usage ratio: 0.42

## Metrics
  - query_length: 11
  - context_length: 86
  - response_length: 46

Structured output inspection:
# Output Inspection

Analysis of output format, safety, and completeness

Timestamp: 2026-07-01 06:43:26.912269

## Evaluations
  - pass: 1.00
    Reasoning: Output is not empty
  - pass: 1.00
    Reasoning: Output matches json format

## Metrics
  - length: 41
  - word_count: 4
✅ PASS: Inspector API smoke tests


In [7]:
# ============================================================
# 7. Analyzer smoke tests
# ============================================================

def test_analyzers():
    from genaiscope.analyzers import (
        CostAnalyzer,
        PIIDetector,
        HallucinationDetector,
        SafetyAnalyzer,
        StructuredOutputValidator,
    )

    print("\nCost analysis:")
    cost_analyzer = CostAnalyzer()
    print(cost_analyzer.estimate_cost("gpt-4", 100, 200))

    print("\nPII detection:")
    pii_detector = PIIDetector()
    pii_text = "My email is john@example.com and phone is 9876543210."
    print("Detections:", pii_detector.detect(pii_text))
    print("Redacted:", pii_detector.redact(pii_text))

    print("\nHallucination detection:")
    hallucination_detector = HallucinationDetector()
    print(hallucination_detector.detect(
        "Bangalore is a city in India.",
        "Bangalore is a city in India and the capital of Karnataka."
    ))

    print("\nSafety analysis:")
    safety_analyzer = SafetyAnalyzer()
    print(safety_analyzer.analyze("Explain AI safety in simple terms."))

    print("\nStructured output validation:")
    validator = StructuredOutputValidator()
    print(validator.validate_json('{"status": "ok", "score": 10}'))

    return "Analyzer tests completed"

test_results.append(safe_call("Analyzer smoke tests", test_analyzers))



Analyzer smoke tests

Cost analysis:
{'input_cost': 0.003, 'output_cost': 0.012, 'total_cost': 0.015}

PII detection:
Detections: {'email': ['john@example.com'], 'phone': ['9876543210']}
Redacted: My email is [EMAIL] and phone is [PHONE].

Hallucination detection:
{'hallucination_risk': 0.5, 'contains_uncertainty': False, 'unsupported_statements': 1}

Safety analysis:
{}

Structured output validation:
{'valid': True, 'data': {'status': 'ok', 'score': 10}}
✅ PASS: Analyzer smoke tests


In [8]:
# ============================================================
# 8. Scoring engine smoke test
# ============================================================

def test_scoring_engine():
    from genaiscope import ScoringEngine

    engine = ScoringEngine()

    print("\nBuilt-in length score:")
    print(engine.score("This is a small test for GenAIScope.", "length"))

    print("\nBuilt-in evaluation:")
    print(engine.evaluate("This is a small test for GenAIScope.", "null_safety", threshold=0.5))

    print("\nCustom scorer:")
    def custom_quality_scorer(text: str) -> float:
        return min(len(text) / 100.0, 1.0)

    engine.register("custom_quality", custom_quality_scorer)
    print(engine.score("This is a custom quality scoring test.", "custom_quality"))

    return "Scoring engine completed"

test_results.append(safe_call("Scoring engine smoke test", test_scoring_engine))



Scoring engine smoke test

Built-in length score:
0.0035003500350035003

Built-in evaluation:
score=1.0 label='pass' reasoning='Score 1.00 vs threshold 0.5' metadata={}

Custom scorer:
0.38
✅ PASS: Scoring engine smoke test


## Local memory tests

These tests validate basic local-first memory behavior: storing facts, searching them, adding prompts, and checking prompt coach output when available.


In [9]:
# ============================================================
# 9. Local memory smoke test
# ============================================================

def test_local_memory():
    from genaiscope.memory import MemoryStore

    memory = MemoryStore()

    print("\nAdding memories...")
    item1 = memory.add("User prefers short CTO-level answers.", memory_type="preference")
    item2 = memory.add("GenAIScope is being tested in Google Colab.", memory_type="project")
    item3 = memory.add("Use local-first memory before production deployment.", memory_type="guideline")

    print("Items:")
    print(item1)
    print(item2)
    print(item3)

    print("\nSearching memory:")
    results = memory.search("answer style CTO")
    print(results)

    if hasattr(memory, "remember"):
        print("\nTesting production-style remember API:")
        remembered = memory.remember(
            "Temporary project context for Colab smoke test.",
            memory_type="temporary",
            user_id="sapan",
            project_id="genaiscope",
            importance=7,
            ttl_days=3,
        )
        print(remembered)

    return "Local memory completed"

test_results.append(safe_call("Local memory smoke test", test_local_memory))



Local memory smoke test

Adding memories...


Items:
id='fc65777f-33d0-48d0-85ed-0b48c902c285' content='User prefers short CTO-level answers.' memory_type='preference' user_id=None workspace_id=None project_id=None agent_id=None session_id=None source='manual' tags=[] metadata={} importance=5 visibility='private' ttl_seconds=None prompt_score=None prompt_risk_level=None prompt_comments=[] prompt_suggestions=[] expires_at=None created_at=datetime.datetime(2026, 7, 1, 6, 43, 26, 938285, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 7, 1, 6, 43, 26, 938285, tzinfo=datetime.timezone.utc)
id='1af9d1bd-a522-4b26-8afd-8d0b9725a6df' content='GenAIScope is being tested in Google Colab.' memory_type='project' user_id=None workspace_id=None project_id=None agent_id=None session_id=None source='manual' tags=[] metadata={} importance=5 visibility='private' ttl_seconds=None prompt_score=None prompt_risk_level=None prompt_comments=[] prompt_suggestions=[] expires_at=None created_at=datetime.datetime(2026, 7, 1, 6, 43, 26, 9434

In [10]:
# ============================================================
# 10. Prompt coach smoke test
# ============================================================

def test_prompt_coach():
    from genaiscope.memory import MemoryStore

    memory = MemoryStore()

    if not hasattr(memory, "add_prompt"):
        print("MemoryStore.add_prompt is not available in this installed version.")
        return "Skipped: add_prompt not available"

    item = memory.add_prompt("Summarize this properly.")
    print("Prompt item:", item)

    for attr in ["prompt_score", "prompt_comments", "prompt_suggestions"]:
        print(f"{attr}:", getattr(item, attr, "not available"))

    return "Prompt coach completed"

test_results.append(safe_call("Prompt coach smoke test", test_prompt_coach))



Prompt coach smoke test
Prompt item: id='c9062a8c-4eaf-4028-b541-bd0297d1b396' content='Summarize this properly.' memory_type='prompt' user_id=None workspace_id=None project_id=None agent_id=None session_id=None source='manual' tags=[] metadata={} importance=5 visibility='private' ttl_seconds=None prompt_score=20 prompt_risk_level='high' prompt_comments=['Prompt is very short and may not provide enough context.', 'Prompt uses vague words: properly.'] prompt_suggestions=['Add a role or persona for the assistant.', 'Specify the expected output format.', 'Add constraints such as length, style, exclusions, or allowed sources.', 'Add success criteria or a quick rubric.'] expires_at=None created_at=datetime.datetime(2026, 7, 1, 6, 43, 26, 963199, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 7, 1, 6, 43, 26, 963199, tzinfo=datetime.timezone.utc)
prompt_score: 20
prompt_comments: ['Prompt is very short and may not provide enough context.', 'Prompt uses vague words: properl

In [11]:
# ============================================================
# 11. Semantic cache smoke test
# ============================================================

def test_semantic_cache():
    from genaiscope.memory import MemoryStore
    from genaiscope.cache import SemanticCache

    memory = MemoryStore()
    cache = SemanticCache(memory_store=memory)

    cache.set(
        prompt="Summarize refund policy",
        response="Refunds are available within 7 days if the booking conditions allow it.",
        user_id="sapan",
    )

    hit = cache.get(
        prompt="Can you summarize the refund policy?",
        user_id="sapan",
    )

    print("Cache hit:", hit)
    return hit

test_results.append(safe_call("Semantic cache smoke test", test_semantic_cache))



Semantic cache smoke test
Cache hit: response='Refunds are available within 7 days if the booking conditions allow it.' score=1.5 memory_id='bdaed931-4111-4d49-a333-1ded19c9974f' model=None metadata={'model': None, 'response': 'Refunds are available within 7 days if the booking conditions allow it.'}
✅ PASS: Semantic cache smoke test


## File memory test

This creates a small Markdown document inside Colab, adds it to GenAIScope file memory, and searches it.


In [12]:
# ============================================================
# 12. File memory smoke test
# ============================================================

def test_file_memory():
    sample_file = WORKDIR / "sample_travel_ai_policy.md"
    sample_file.write_text(
        """# Travel AI Policy

GenAIScope should help inspect prompts, memory, traces, cost, PII risk, and dashboard readiness.

Refund policy:
Customers may request refunds based on supplier rules, cancellation date, and payment status.

Security policy:
Do not expose API keys, payment data, passport data, or personal contact details in logs.
""",
        encoding="utf-8",
    )

    from genaiscope.files import FileMemory

    files = FileMemory()
    added = files.add_file(str(sample_file))
    print("Added file:", added)

    print("\nSearch result for 'refund policy':")
    print(files.search("refund policy"))

    print("\nSearch result for 'API keys security':")
    print(files.search("API keys security"))

    return "File memory completed"

test_results.append(safe_call("File memory smoke test", test_file_memory))



File memory smoke test
Added file: [MemoryItem(id='a5e08480-c32d-4c23-aed9-42d774186a0a', content='# Travel AI Policy\n\nGenAIScope should help inspect prompts, memory, traces, cost, PII risk, and dashboard readiness.\n\nRefund policy:\nCustomers may request refunds based on supplier rules, cancellation date, and payment status.\n\nSecurity policy:\nDo not expose API keys, payment data, passport data, or personal contact details in logs.', memory_type='document', user_id=None, workspace_id=None, project_id=None, agent_id=None, session_id=None, source='file', tags=[], metadata={'chunk_index': 0, 'file_name': 'sample_travel_ai_policy.md', 'file_path': '/tmp/genaiscope_colab_test/sample_travel_ai_policy.md', 'file_size': 335, 'file_type': '.md', 'indexed_at': '2026-07-01T06:43:26.993022+00:00', 'invalid_json': False, 'loader': 'md', 'total_chunks': 1}, importance=5, visibility='private', ttl_seconds=None, prompt_score=None, prompt_risk_level=None, prompt_comments=[], prompt_suggestions=[

## Local tracing test

This logs a local trace without calling any external LLM.


In [13]:
# ============================================================
# 13. Local tracing smoke test
# ============================================================

def test_local_tracing():
    from genaiscope.tracing import LocalTracer

    tracer = LocalTracer()
    result = tracer.log(
        name="colab-demo-call",
        input_text="hello",
        output_text="hi",
        model="local-test-model",
        input_tokens=5,
        output_tokens=2,
        estimated_cost=0.0,
    )

    print("Trace log result:", result)
    print("Tracer attributes:", [x for x in dir(tracer) if not x.startswith("_")][:80])

    return "Local tracing completed"

test_results.append(safe_call("Local tracing smoke test", test_local_tracing))



Local tracing smoke test
Trace log result: id='b692b58e-ef1a-43d6-bfca-57daf930f59c' name='colab-demo-call' input_text='hello' output_text='hi' model='local-test-model' provider=None input_tokens=5 output_tokens=2 estimated_cost=0.0 latency_ms=None status='success' error=None metadata={} created_at=datetime.datetime(2026, 7, 1, 6, 43, 27, 5823, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 7, 1, 6, 43, 27, 5823, tzinfo=datetime.timezone.utc)
Tracer attributes: ['clear', 'close', 'exporters', 'get', 'list', 'log', 'stats', 'store', 'trace']
✅ PASS: Local tracing smoke test


## CLI tests

The following cell checks the command-line interface. CLI commands are allowed to fail without stopping the notebook, because command names can change between releases. The printed output will show what is available in your installed version.


In [14]:
# ============================================================
# 14. CLI command smoke tests
# ============================================================

def test_cli_commands():
    sample_file = WORKDIR / "sample_travel_ai_policy.md"
    if not sample_file.exists():
        sample_file.write_text("GenAIScope test file with refund policy and API key safety.", encoding="utf-8")

    commands = [
        ["genaiscope", "--help"],
        ["genaiscope", "version"],
        ["genaiscope", "config-show"],
        ["genaiscope", "inspect-prompt", "What is AI?"],
        ["genaiscope", "detect-pii", "My email is john@example.com"],
        ["genaiscope", "detect-pii", "Email: john@example.com", "--redact"],
        ["genaiscope", "estimate-cost", "gpt-4", "100", "200"],
        ["genaiscope", "validate-output", '{"test": "data"}', "--format", "json"],
        ["genaiscope", "memory", "add", "User prefers concise answers", "--type", "preference"],
        ["genaiscope", "memory", "add-prompt", "Summarize this properly."],
        ["genaiscope", "memory", "search", "concise answers"],
        ["genaiscope", "files", "add", str(sample_file)],
        ["genaiscope", "trace", "stats"],
        ["genaiscope", "dashboard", "generate"],
    ]

    cli_summary = []
    for cmd in commands:
        # cwd=WORKDIR keeps CLI side effects (its default .genaiscope/ db, dashboard html, etc.)
        # out of the repo working copy, even when INSTALL_SOURCE == "local".
        result = run_command(cmd, cwd=WORKDIR, check=False, timeout=90)
        cli_summary.append({
            "command": " ".join(cmd),
            "returncode": result.returncode,
        })

    print("\nCLI summary:")
    print(json.dumps(cli_summary, indent=2))
    return cli_summary

test_results.append(safe_call("CLI command smoke tests", test_cli_commands))



CLI command smoke tests

$ genaiscope --help


                                                                                
 Usage: genaiscope [OPTIONS] COMMAND [ARGS]...                                  
                                                                                
 GenAIScope: Inspect, test, secure, optimize, and operationalize GenAI          
 applications.                                                                  
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ────────────────

GenAIScope version 0.7.0


$ genaiscope config-show


          GenAIScope Configuration          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting         ┃ Value                  ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ provider        │ openai                 │
│ openai_model    │ gpt-4                  │
│ anthropic_model │ claude-3-opus-20240229 │
│ google_model    │ gemini-pro             │
│ max_tokens      │ 2048                   │
│ temperature     │ 0.7                    │
│ timeout         │ 30                     │
│ retries         │ 3                      │
│ log_level       │ INFO                   │
└─────────────────┴────────────────────────┘


$ genaiscope inspect-prompt What is AI?


# Prompt Inspection

Analysis of prompt quality and potential issues

Timestamp: 2026-07-01 06:43:28.582315

## Evaluations
  - pass: 0.80
    Reasoning: Prompt structure looks reasonable

## Metrics
  - length: 11
  - word_count: 3
  - avg_word_length: 3.0

2026-07-01 12:13:28,582 - genaiscope.inspect - INFO - Inspecting prompt: What is AI?...


$ genaiscope detect-pii My email is john@example.com


...

╭───────────────── Prompt stored ─────────────────╮
│ Memory ID: 5a0ccaaf-f426-42b3-863d-fcb553d5cd4f │
│ Prompt Score: 20                                │
│ Risk Level: high                                │
╰─────────────────────────────────────────────────╯
Comment: Prompt is very short and may not provide enough context.
Comment: Prompt uses vague words: properly.
Suggestion: Add a role or persona for the assistant.
Suggestion: Specify the expected output format.
Suggestion: Add constraints such as length, style, exclusions, or allowed 
sources.
Suggestion: Add success criteria or a quick rubric.


$ genaiscope memory search concise answers


                           Memory Search Results                            
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━┓
┃ Score ┃ Type       ┃ Content                               ┃ Vec  ┃ KW   ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━┩
│ 1.25  │ preference │ User prefers concise answers          │ 0.00 │ 1.00 │
│ 1.25  │ preference │ User prefers concise answers          │ 0.00 │ 1.00 │
│ 0.39  │ preference │ User prefers short CTO-level answers. │ 0.00 │ 0.14 │
└───────┴────────────┴───────────────────────────────────────┴──────┴──────┘


$ genaiscope files add /tmp/genaiscope_colab_test/sample_travel_ai_policy.md


Indexed 1 chunks


$ genaiscope trace stats


{
  "total_traces": 1,
  "success_count": 1,
  "error_count": 0,
  "total_estimated_cost": 0.0,
  "average_latency_ms": null,
  "total_input_tokens": 5,
  "total_output_tokens": 2,
  "traces_by_model": {
    "local-test-model": 1
  },
  "traces_by_provider": {}
}


$ genaiscope dashboard generate


Dashboard generated: .genaiscope/reports/dashboard.html


CLI summary:
[
  {
    "command": "genaiscope --help",
    "returncode": 0
  },
  {
    "command": "genaiscope version",
    "returncode": 0
  },
  {
    "command": "genaiscope config-show",
    "returncode": 0
  },
  {
    "command": "genaiscope inspect-prompt What is AI?",
    "returncode": 0
  },
  {
    "command": "genaiscope detect-pii My email is john@example.com",
    "returncode": 0
  },
  {
    "command": "genaiscope detect-pii Email: john@example.com --redact",
    "returncode": 0
  },
  {
    "command": "genaiscope estimate-cost gpt-4 100 200",
    "returncode": 0
  },
  {
    "command": "genaiscope validate-output {\"test\": \"data\"} --format json",
    "returncode": 0
  },
  {
    "command": "genaiscope memory add User prefers concise answers --type preference",
    "returncode": 0
  },
  {
    "command": "genaiscope memory add-prompt Summarize this properly.",
    "returncode": 0
  },
  {
    "command": "genaiscop

## Dashboard output check

If `genaiscope dashboard generate` created an HTML dashboard, this cell finds it and displays a link inside Colab.


In [15]:
# ============================================================
# 15. Find generated dashboard HTML files
# ============================================================

def find_dashboard_outputs():
    search_roots = [pathlib.Path.cwd(), WORKDIR, pathlib.Path("/content")]
    html_files = []

    for root in search_roots:
        if root.exists():
            try:
                html_files.extend(root.rglob("*.html"))
            except Exception:
                pass

    # Deduplicate and sort by latest modified time
    unique_files = sorted(set(html_files), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

    print("HTML files found:")
    for p in unique_files[:20]:
        print("-", p)

    if unique_files:
        try:
            from IPython.display import display, HTML
            latest = unique_files[0]
            print("\nLatest dashboard candidate:", latest)
            display(HTML(f'<a href="file://{latest}" target="_blank">Open latest HTML file: {latest.name}</a>'))
        except Exception as exc:
            print("Could not display HTML link:", exc)

    return [str(p) for p in unique_files[:20]]

test_results.append(safe_call("Find dashboard outputs", find_dashboard_outputs))



Find dashboard outputs
HTML files found:
- /tmp/genaiscope_colab_test/.genaiscope/reports/dashboard.html
- /tmp/genaiscope_colab_test/context_doctor_report.html

Latest dashboard candidate: /tmp/genaiscope_colab_test/.genaiscope/reports/dashboard.html


✅ PASS: Find dashboard outputs


## Context Doctor smoke test (v0.6.0)

Exercises the `GenAIScope` facade end to end: memory, context builder, diagnosis, cost,
router, analytics, and the HTML report — no LLM call required.


In [16]:
# ============================================================
# 16. Context Doctor smoke test
# ============================================================

def test_context_doctor():
    from genaiscope import GenAIScope

    db_path = str(WORKDIR / "context_doctor_colab_test.db")
    scope = GenAIScope(db_path=db_path)

    scope.memory.add(
        memory_type="profile_memory",
        content="Colab tester is a CTO evaluating GenAIScope v0.7.0.",
        tags=["profile", "cto"],
    )

    weak_prompt = "Write answer for feature velocity."
    context = scope.context.build(weak_prompt, top_k=5)
    print("Improved prompt:", context.improved_prompt)

    response = "Feature velocity is how fast a team ships value."
    report = scope.doctor.diagnose(prompt=weak_prompt, response=response, memories_used=context.retrieved_memories)
    print("Health score:", report.context_health_score)
    print("Missing context:", report.missing_context)

    print("Cost estimate:", scope.cost.estimate("openai", "gpt-4o-mini", 100, 200))
    print("Router recommendation:", scope.router.recommend(weak_prompt))

    with scope.trace(name="colab-context-doctor", model="local") as trace:
        trace.log(prompt=weak_prompt, response=response)

    print("Usage summary:", scope.analytics.usage_summary())

    report_path = WORKDIR / "context_doctor_report.html"
    scope.report.generate_html(str(report_path))
    print("HTML report written:", report_path, "exists:", report_path.exists())

    return report.context_health_score

test_results.append(safe_call("Context Doctor smoke test", test_context_doctor))



Context Doctor smoke test
Improved prompt: Write answer for feature velocity. Keep it concise and well-structured. Use a senior, professional tone. Connect it to business impact and outcomes. Write for the intended audience explicitly.
Health score: 47
Missing context: ['Target audience', 'Desired answer length or format', 'Tone preference', 'Business context (impact, outcomes)']
Cost estimate: provider='openai' model='gpt-4o-mini' input_tokens=100 output_tokens=200 input_cost=1.4999999999999999e-05 output_cost=0.00011999999999999999 total_cost=0.00013499999999999997 priced=True
Router recommendation: recommended_model_type='writing' reason='The prompt asks for original written content.' suggested_providers=['openai', 'anthropic', 'google'] cost_sensitivity='medium'
Usage summary: total_requests=4 total_input_tokens=0 total_output_tokens=0 total_tokens=0 total_estimated_cost=0.0 average_latency_ms=0.1332809997620643 cost_by_provider={'unknown': 0.0} cost_by_model={'local': 0.0} tokens

## MCP tools smoke test (v0.7.0)

Calls the MCP tool functions directly (no MCP client/transport needed). The three
trace-dependent tools (`analytics_usage_summary`, `analytics_prompt_patterns`,
`report_generate`) are expected to return a plain `{"error": ...}` dict, not raise, when no
tracer is supplied — that's the documented v0.7.0 behavior, not a bug.


In [17]:
# ============================================================
# 17. MCP tools smoke test
# ============================================================

def test_mcp_tools():
    from genaiscope.memory import MemoryStore
    from genaiscope.mcp import tools as mcp_tools

    db_path = str(WORKDIR / "mcp_tools_colab_test.db")
    memory = MemoryStore(db_path=db_path)

    remembered = mcp_tools.tool_memory_remember(
        memory, content="MCP smoke test memory", memory_type="note", tags=["mcp", "colab"]
    )
    print("memory_remember:", remembered)

    search_result = mcp_tools.tool_memory_search(memory, query="MCP smoke test")
    print("memory_search:", search_result)

    diagnosis = mcp_tools.tool_doctor_diagnose(memory, prompt="Write something", response="Something.")
    print("doctor_diagnose score:", diagnosis.get("context_health_score"))

    # No tracer supplied -- trace-dependent tools must return an error dict, not raise.
    usage_no_tracer = mcp_tools.tool_analytics_usage_summary(None)
    assert "error" in usage_no_tracer, f"expected error dict without tracer, got {usage_no_tracer}"
    print("analytics_usage_summary (no tracer):", usage_no_tracer)

    return "MCP tools completed"

test_results.append(safe_call("MCP tools smoke test", test_mcp_tools))



MCP tools smoke test
memory_remember: {'id': 'a361f0d9-4974-4c6c-9d2e-6ec5c51223e3', 'memory_type': 'note', 'content': 'MCP smoke test memory'}
memory_search: {'results': [{'id': 'a361f0d9-4974-4c6c-9d2e-6ec5c51223e3', 'content': 'MCP smoke test memory', 'score': 1.25, 'match_type': 'exact', 'ranking_reason': 'Vector similarity 0.00 + keyword match (mcp, smoke, test) 1.00, boosted by recency.'}, {'id': '66fcea54-90dd-4445-8653-99f0bc9db1c1', 'content': 'MCP smoke test memory', 'score': 1.25, 'match_type': 'exact', 'ranking_reason': 'Vector similarity 0.00 + keyword match (mcp, smoke, test) 1.00, boosted by recency.'}, {'id': '643bade0-c1b3-4b81-a3a9-eb214f9cf2da', 'content': 'MCP smoke test memory', 'score': 1.25, 'match_type': 'exact', 'ranking_reason': 'Vector similarity 0.00 + keyword match (mcp, smoke, test) 1.00, boosted by recency.'}, {'id': '43087624-9a37-4810-92fc-18d889a74ff5', 'content': 'MCP smoke test memory', 'score': 1.25, 'match_type': 'exact', 'ranking_reason': 'Vector

## Live LLM gateway smoke test (v0.7.0)

`scope.gateway.ask()` auto-routes to a real OpenAI/Anthropic/Google call. This cell only
performs a live call if it finds a provider API key in the environment (`OPENAI_API_KEY`,
`ANTHROPIC_API_KEY`, or `GOOGLE_API_KEY`/`GEMINI_API_KEY`); otherwise it verifies the
documented `GatewayError` is raised when every candidate provider fails/is unavailable.


In [18]:
# ============================================================
# 18. Live LLM gateway smoke test
# ============================================================

def test_gateway():
    from genaiscope import GenAIScope
    from genaiscope.core.errors import GatewayError

    db_path = str(WORKDIR / "gateway_colab_test.db")
    scope = GenAIScope(db_path=db_path)

    has_key = any(
        os.environ.get(k) for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GEMINI_API_KEY"]
    )

    if not has_key:
        print("No provider API key found in environment -- verifying GatewayError path only.")
        try:
            scope.gateway.complete("Say hello", provider="auto")
            raise AssertionError("expected GatewayError when no provider key is configured")
        except GatewayError as exc:
            print("Got expected GatewayError:", exc)
            return "GatewayError path verified (no API key present)"

    print("Provider API key found -- making a real live call.")
    result = scope.gateway.complete("Say hello in exactly three words.", provider="auto")
    print("Gateway response:", result)
    return result

test_results.append(safe_call("Live LLM gateway smoke test", test_gateway))



Live LLM gateway smoke test
No provider API key found in environment -- verifying GatewayError path only.


/home/sap-ai-plug/CODE/GenAIScope/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sap-ai-plug/CODE/GenAIScope/src/genaiscope/adapters/gemini_adapter.py:30: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


Got expected GatewayError: All candidate providers failed: ['openai', 'anthropic', 'google']. Last error: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.
✅ PASS: Live LLM gateway smoke test


## Cross-encoder reranking smoke test (v0.7.0)

`rerank=True` reranks hybrid-search candidates with `cross-encoder/ms-marco-MiniLM-L-6-v2`
(from `sentence-transformers`). The first call downloads the model from Hugging Face, so
this cell needs outbound network access and is skipped gracefully if that's unavailable.


In [19]:
# ============================================================
# 19. Cross-encoder reranking smoke test
# ============================================================

def test_reranking():
    from genaiscope.memory import MemoryStore

    db_path = str(WORKDIR / "rerank_colab_test.db")
    memory = MemoryStore(db_path=db_path)
    memory.add("The refund policy allows returns within 30 days.", memory_type="policy")
    memory.add("Our office is located in Bangalore, India.", memory_type="fact")
    memory.add("Customers can request a refund if the item is unused.", memory_type="policy")

    try:
        results = memory.search("refund policy", rerank=True)
    except Exception as exc:
        print("Reranking unavailable in this environment (likely no network for model download):", exc)
        return "Skipped: reranking model unavailable"

    print("Reranked results:", results)
    return results

test_results.append(safe_call("Cross-encoder reranking smoke test", test_reranking))



Cross-encoder reranking smoke test


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4696.37it/s]

Reranked results: [MemorySearchResult(item=MemoryItem(id='3b28dcb8-72c4-487e-9504-d2aabe7dfa56', content='The refund policy allows returns within 30 days.', memory_type='policy', user_id=None, workspace_id=None, project_id=None, agent_id=None, session_id=None, source='manual', tags=[], metadata={}, importance=5, visibility='private', ttl_seconds=None, prompt_score=None, prompt_risk_level=None, prompt_comments=[], prompt_suggestions=[], expires_at=None, created_at=datetime.datetime(2026, 7, 1, 6, 43, 46, 635568, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 7, 1, 6, 43, 46, 635568, tzinfo=datetime.timezone.utc)), score=1.0, match_type='exact', matched_terms=['policy', 'refund'], ranking_reason='cross_encoder_rerank', keyword_score=1.0, vector_score=0.0, fused_score=1.0, embedder_name='none'), MemorySearchResult(item=MemoryItem(id='d52ba698-928d-4989-a571-c6946429154b', content='The refund policy allows returns within 30 days.', memory_type='policy', user_id=None, wor

## Agent evaluation smoke test (v0.7.0)

`genaiscope.evals.run_agent_eval` scores a multi-step agent trajectory against a
user-provided callable. Library API only -- no CLI command.


In [20]:
# ============================================================
# 20. Agent evaluation smoke test
# ============================================================

def test_agent_eval():
    from genaiscope.evals import run_agent_eval, AgentTrajectory, AgentStep

    def agent_fn(name: str, args: dict) -> str:
        if name == "boom":
            raise RuntimeError("simulated tool failure")
        return f"{args.get('text', '')}".upper()

    trajectory = AgentTrajectory(
        task="colab uppercase-echo agent",
        steps=[
            AgentStep(name="step1", args={"text": "hello"}, expected_output="HELLO"),
            AgentStep(name="step2", args={"text": "world"}, expected_output="WORLD"),
        ],
    )

    report = run_agent_eval(trajectory, agent_fn)
    print("Agent eval report:", report)
    assert report.steps_total == 2
    assert report.steps_passed == 2
    return report.step_completion_rate

test_results.append(safe_call("Agent evaluation smoke test", test_agent_eval))



Agent evaluation smoke test
Agent eval report: task='colab uppercase-echo agent' task_id='5a725c82-026e-49ba-b5f4-9e007d883bad' steps_total=2 steps_passed=2 step_completion_rate=1.0 total_latency_ms=0.007 step_results=[AgentStepResult(name='step1', passed=True, actual_output='HELLO', latency_ms=0.005, error=None), AgentStepResult(name='step2', passed=True, actual_output='WORLD', latency_ms=0.002, error=None)]
✅ PASS: Agent evaluation smoke test


## LangChain integration smoke test (v0.7.0)

`GenAIScopeChatMessageHistory` implements `langchain_core.chat_history.BaseChatMessageHistory`
backed by a GenAIScope `MemoryStore`.


In [21]:
# ============================================================
# 21. LangChain integration smoke test
# ============================================================

def test_langchain_integration():
    from genaiscope.memory import MemoryStore
    from genaiscope.integrations.langchain import GenAIScopeChatMessageHistory
    from langchain_core.messages import HumanMessage, AIMessage

    db_path = str(WORKDIR / "langchain_colab_test.db")
    memory = MemoryStore(db_path=db_path)
    history = GenAIScopeChatMessageHistory(store=memory, session_id="colab-session")

    history.add_messages([HumanMessage(content="Hello there"), AIMessage(content="Hi! How can I help?")])

    print("LangChain history messages:", history.messages)
    return [type(m).__name__ for m in history.messages]

test_results.append(safe_call("LangChain integration smoke test", test_langchain_integration))



LangChain integration smoke test
LangChain history messages: [HumanMessage(content='Hello there', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hello there', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hello there', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hello there', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
✅ PASS: LangChain integration smoke test


## LlamaIndex integration smoke test (v0.7.0)

`GenAIScopeMemory` implements `llama_index.core.memory.types.BaseMemory` backed by a
GenAIScope `MemoryStore`.


In [22]:
# ============================================================
# 22. LlamaIndex integration smoke test
# ============================================================

def test_llamaindex_integration():
    from genaiscope.memory import MemoryStore
    from genaiscope.integrations.llamaindex import GenAIScopeMemory
    from llama_index.core.base.llms.types import ChatMessage, MessageRole

    db_path = str(WORKDIR / "llamaindex_colab_test.db")
    memory_store = MemoryStore(db_path=db_path)
    llama_memory = GenAIScopeMemory.from_defaults(store=memory_store, session_id="colab-session")

    llama_memory.put(ChatMessage(role=MessageRole.USER, content="Hello there"))
    llama_memory.put(ChatMessage(role=MessageRole.ASSISTANT, content="Hi! How can I help?"))

    chat_history = llama_memory.get()
    print("LlamaIndex chat history:", chat_history)
    return [m.role for m in chat_history]

test_results.append(safe_call("LlamaIndex integration smoke test", test_llamaindex_integration))



LlamaIndex integration smoke test


LlamaIndex chat history: [ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hello there')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hi! How can I help?')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hello there')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hi! How can I help?')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hello there')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hi! How can I help?')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Hello there')]), ChatMessage(role=<MessageRole.A

## Langfuse batch export smoke test (v0.7.0)

`genaiscope export --format langfuse` writes a JSON file matching Langfuse's documented
`POST /api/public/ingestion` batch-event shape. No network call, no `langfuse` dependency.


In [23]:
# ============================================================
# 23. Langfuse batch export smoke test
# ============================================================

def test_langfuse_export():
    from genaiscope.tracing import LocalTracer
    from genaiscope.export import export_langfuse

    db_path = str(WORKDIR / "langfuse_export_colab_test.db")
    tracer = LocalTracer(db_path=db_path)
    tracer.log(
        name="colab-langfuse-demo",
        input_text="hello",
        output_text="hi",
        model="local",
        input_tokens=5,
        output_tokens=2,
        estimated_cost=0.0,
    )

    out_path = WORKDIR / "langfuse_export.json"
    export_langfuse(tracer, str(out_path))

    payload = json.loads(out_path.read_text())
    print("Langfuse batch keys:", list(payload.keys()) if isinstance(payload, dict) else type(payload))
    print("Sample:", json.dumps(payload, indent=2)[:1000])
    return payload if isinstance(payload, dict) else str(type(payload))

test_results.append(safe_call("Langfuse batch export smoke test", test_langfuse_export))



Langfuse batch export smoke test
Langfuse batch keys: ['batch']
Sample: {
  "batch": [
    {
      "id": "8118f3eb-f8e9-4e36-adf6-98784b23ae1b-trace",
      "type": "trace-create",
      "timestamp": "2026-07-01T06:44:03.250688+00:00",
      "body": {
        "id": "8118f3eb-f8e9-4e36-adf6-98784b23ae1b",
        "name": "colab-langfuse-demo",
        "input": "hello",
        "output": "hi",
        "metadata": {},
        "timestamp": "2026-07-01T06:44:03.250688+00:00"
      }
    },
    {
      "id": "8118f3eb-f8e9-4e36-adf6-98784b23ae1b-generation",
      "type": "generation-create",
      "timestamp": "2026-07-01T06:44:03.250688+00:00",
      "body": {
        "id": "8118f3eb-f8e9-4e36-adf6-98784b23ae1b-generation",
        "traceId": "8118f3eb-f8e9-4e36-adf6-98784b23ae1b",
        "name": "colab-langfuse-demo",
        "startTime": "2026-07-01T06:44:03.250688+00:00",
        "endTime": "2026-07-01T06:44:03.250688+00:00",
        "model": "local",
        "input": "hello",
       

## OpenTelemetry exporter smoke test (v0.7.0)

`LocalTracer(exporters=[OTelExporter()])` maps every logged trace to a real OTel span using
`gen_ai.*` semantic-convention attributes, through whatever global `TracerProvider` is
configured. An exporter failure must never break local tracing.


In [24]:
# ============================================================
# 24. OpenTelemetry exporter smoke test
# ============================================================

def test_otel_exporter():
    from opentelemetry import trace as otel_trace
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

    from genaiscope.tracing import LocalTracer
    from genaiscope.integrations.otel import OTelExporter

    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
    otel_trace.set_tracer_provider(provider)

    db_path = str(WORKDIR / "otel_colab_test.db")
    tracer = LocalTracer(db_path=db_path, exporters=[OTelExporter()])
    result = tracer.log(
        name="colab-otel-demo",
        input_text="hello",
        output_text="hi",
        model="local",
        input_tokens=5,
        output_tokens=2,
        estimated_cost=0.0,
    )

    print("Local trace result (OTel span printed above via ConsoleSpanExporter):", result)
    return "OTel exporter completed"

test_results.append(safe_call("OpenTelemetry exporter smoke test", test_otel_exporter))



OpenTelemetry exporter smoke test
{
    "name": "colab-otel-demo",
    "context": {
        "trace_id": "0xce3a4fd076ddad30feb2e89c8df23d35",
        "span_id": "0xd5a0357e6bb15e2b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-01T06:44:03.286208Z",
    "end_time": "2026-07-01T06:44:03.286208Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "genaiscope.trace_id": "8fb9e45e-fa22-4c89-b246-97b3f173cb9d",
        "gen_ai.request.model": "local",
        "gen_ai.usage.input_tokens": 5,
        "gen_ai.usage.output_tokens": 2,
        "genaiscope.estimated_cost": 0.0
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "cefa7fbe-b134-4a4e-b081-16f768ac1560",
            "servic

Local trace result (OTel span printed above via ConsoleSpanExporter): id='8fb9e45e-fa22-4c89-b246-97b3f173cb9d' name='colab-otel-demo' input_text='hello' output_text='hi' model='local' provider=None input_tokens=5 output_tokens=2 estimated_cost=0.0 latency_ms=None status='success' error=None metadata={} created_at=datetime.datetime(2026, 7, 1, 6, 44, 3, 286208, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 7, 1, 6, 44, 3, 286208, tzinfo=datetime.timezone.utc)
✅ PASS: OpenTelemetry exporter smoke test


## Browser extension structure check (v0.7.0)

The browser extension can't run inside a Jupyter kernel (it needs a real browser), so this
cell only verifies the expected files exist in `browser-extension/` -- a manifest and content
scripts that talk to the existing `/v1/prompts` and `/v1/memory/remember` REST endpoints.


In [25]:
# ============================================================
# 25. Browser extension structure check
# ============================================================

def check_browser_extension():
    ext_dir = REPO_DIR / "browser-extension"
    if not ext_dir.exists():
        return "Skipped: browser-extension/ not found (check REPO_DIR)"

    manifest = ext_dir / "manifest.json"
    contents = sorted(p.name for p in ext_dir.iterdir())
    print("browser-extension/ contents:", contents)

    assert manifest.exists(), "manifest.json missing from browser-extension/"
    manifest_data = json.loads(manifest.read_text())
    print("Manifest name/version:", manifest_data.get("name"), manifest_data.get("version"))

    return contents

test_results.append(safe_call("Browser extension structure check", check_browser_extension))



Browser extension structure check
browser-extension/ contents: ['README.md', 'background.js', 'content.js', 'manifest.json', 'popup.html', 'popup.js']
Manifest name/version: GenAIScope Capture 0.1.0
✅ PASS: Browser extension structure check


## Run repository tests

This runs `pytest` only when a `tests/` folder exists in the cloned repository.


In [26]:
# ============================================================
# 16. Run pytest tests from repository
# ============================================================

def run_repository_tests():
    if not REPO_DIR.exists():
        print("Repository folder not available. Skipping pytest.")
        return "Skipped: no repo folder"

    tests_dir = REPO_DIR / "tests"
    if not tests_dir.exists():
        print("No tests/ folder found. Skipping pytest.")
        return "Skipped: no tests folder"

    result = run_command([sys.executable, "-m", "pytest", str(tests_dir), "-q"], cwd=REPO_DIR, check=False, timeout=300)
    return {
        "returncode": result.returncode,
        "stdout_tail": result.stdout[-2000:] if result.stdout else "",
        "stderr_tail": result.stderr[-2000:] if result.stderr else "",
    }

test_results.append(safe_call("Repository pytest run", run_repository_tests))



Repository pytest run

$ /home/sap-ai-plug/CODE/GenAIScope/.venv/bin/python3 -m pytest /home/sap-ai-plug/CODE/GenAIScope/tests -q


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.4.2, pluggy-1.6.0
rootdir: /home/sap-ai-plug/CODE/GenAIScope
configfile: pyproject.toml
plugins: cov-5.0.0, anyio-4.13.0, langsmith-0.9.2, asyncio-0.26.0
asyncio: mode=Mode.AUTO, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 210 items

tests/test_adapter_anthropic_mock.py ....                                [  1%]
tests/test_adapter_gemini_mock.py ....                                   [  3%]
tests/test_adapter_openai_mock.py .....s                                 [  6%]
tests/test_adapter_summarizers.py ...sss                                 [  9%]
tests/test_agent_eval.py ....                                            [ 11%]
tests/test_analytics_patterns.py ...                                     [ 12%]
tests/test_analytics_usage.py ...                                        [ 14%]
tests/test_analyzers.py .....      

## Build package check

This optional check verifies whether the package can be built cleanly. It is useful before a GitHub release or PyPI upload.


In [27]:
# ============================================================
# 17. Optional build check
# ============================================================

RUN_BUILD_CHECK = False  # Change to True when you want to test package build

def run_build_check():
    if not RUN_BUILD_CHECK:
        print("Build check disabled. Set RUN_BUILD_CHECK = True to run it.")
        return "Skipped: build check disabled"

    if not REPO_DIR.exists():
        return "Skipped: no repo folder"

    run_command([sys.executable, "-m", "pip", "install", "-U", "build", "twine"], check=False, timeout=240)

    dist_dir = REPO_DIR / "dist"
    if dist_dir.exists():
        shutil.rmtree(dist_dir)

    build_result = run_command([sys.executable, "-m", "build"], cwd=REPO_DIR, check=False, timeout=300)
    twine_result = run_command([sys.executable, "-m", "twine", "check", "dist/*"], cwd=REPO_DIR, check=False, timeout=180)

    return {
        "build_returncode": build_result.returncode,
        "twine_returncode": twine_result.returncode,
    }

test_results.append(safe_call("Optional build check", run_build_check))



Optional build check
Build check disabled. Set RUN_BUILD_CHECK = True to run it.
✅ PASS: Optional build check


## Final validation report

Use this report to decide whether GenAIScope is ready to push, tag, or publish.


In [28]:
# ============================================================
# 18. Final validation report
# ============================================================

import pandas as pd
from IPython.display import display

df = pd.DataFrame(test_results)
display(df)

pass_count = int((df["status"] == "PASS").sum())
fail_count = int((df["status"] == "FAIL").sum())

print(f"PASS: {pass_count}")
print(f"FAIL: {fail_count}")

if fail_count == 0:
    print("\n✅ GenAIScope basic Colab validation completed successfully.")
else:
    print("\n⚠️ Some checks failed. Review the error column and command outputs above.")
    print("Typical causes:")
    print("- API changed between versions")
    print("- CLI command renamed")
    print("- Optional dependencies missing")
    print("- Dashboard path changed")
    print("- Local GitHub main has a temporary issue")


,name,status,value,error
0,Import package and print version,PASS,'0.7.0',
1,Inspect package modules,PASS,"[{'name': 'adapters', 'is_package': True}, {'n...",
2,Inspector API smoke tests,PASS,'Inspector API completed',
3,Analyzer smoke tests,PASS,'Analyzer tests completed',
4,Scoring engine smoke test,PASS,'Scoring engine completed',
5,Local memory smoke test,PASS,'Local memory completed',
6,Prompt coach smoke test,PASS,'Prompt coach completed',
7,Semantic cache smoke test,PASS,CacheHit(response='Refunds are available withi...,
8,File memory smoke test,PASS,'File memory completed',
9,Local tracing smoke test,PASS,'Local tracing completed',


PASS: 24
FAIL: 0

✅ GenAIScope basic Colab validation completed successfully.


## Suggested release checklist after successful Colab test

Run these locally in your project folder after the notebook passes:

```bash
git status
pytest tests/ -v
python -m build
twine check dist/*
git tag v0.3.1
git push origin main --tags
```

For PyPI publishing:

```bash
python -m twine upload dist/*
```

For TestPyPI:

```bash
python -m twine upload --repository testpypi dist/*
```

Do not publish until the Colab notebook, local tests, and package build all pass.
